# Capital Bikeshare Demand Analysis and Prediction

# Executive Summary

This project analyzes daily Capital Bikeshare ridership to identify the factors that most strongly influence demand and develop a predictive model capable of forecasting total daily rentals. Using weather, seasonal, and calendar-related variables, this analysis applies multiple linear regression to evaluate demand drivers while emphasizing model interpretability and practical business insights.

## Key Findings

- The final predictive model explains approximately **81%** of the variation in daily ridership (**R² ≈ 0.81**), demonstrating strong overall performance.
- **Temperature, weather conditions, seasonality, and long-term ridership growth** are the most influential factors affecting daily bike demand.
- Exploratory analysis identified **severe multicollinearity** between the temperature and "feels like" temperature variables. A reduced model was developed to eliminate this issue while maintaining nearly identical predictive performance.
- The final reduced model provides a more stable and interpretable solution, sacrificing only a small amount of predictive accuracy in exchange for substantially improved coefficient reliability.

## Business Recommendations

- Use weather forecasts to proactively adjust bicycle inventory and station capacity.
- Schedule maintenance and redistribution activities during predictable seasonal demand slowdowns.
- Develop targeted marketing strategies for casual and registered riders based on observed usage patterns.
- Continue monitoring long-term ridership trends to support infrastructure planning and future system expansion.

This project demonstrates a complete business analytics workflow, including exploratory data analysis, feature engineering, model development, diagnostic testing, multicollinearity assessment, model refinement, and actionable business recommendations.

## Business Problem

Capital Bikeshare needs to understand the factors that influence daily bicycle demand so that it can improve fleet availability, membership conversion, pricing decisions, and operational planning.

## Objectives

This analysis will:

1. Examine seasonal and weather-related patterns in daily ridership.
2. Compare demand among casual and registered riders.
3. Identify the variables most strongly associated with total rentals.
4. Build and evaluate regression models for predicting daily demand.
5. Translate the findings into operational and business recommendations.

In [60]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Visualization
import plotly.express as px
import plotly.graph_objects as go

# Statistical modeling
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Machine learning
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

pd.set_option("display.max_columns", None)

In [61]:
df = pd.read_csv(
    "/kaggle/input/datasets/ericfrederic/capbike/capital_bike_share_clean.csv"
)

In [62]:
# Standardize column names
df.columns = df.columns.str.strip().str.upper()

# Convert DATE to datetime
df["DATE"] = pd.to_datetime(df["DATE"], format="%m/%d/%Y")

# Create the target variable
df["COUNT"] = df["CASUAL"] + df["REGISTERED"]

# Extract calendar features
df["MONTH"] = df["DATE"].dt.month
df["MONTH_NAME"] = df["DATE"].dt.month_name()
df["YEAR"] = df["DATE"].dt.year

# Create bad-weather indicator
df["BADWEATHER"] = np.where(
    df["WEATHERSIT"].isin([3, 4]),
    "YES",
    "NO"
)

# Set month order for plotting
month_order = [
    "January", "February", "March", "April",
    "May", "June", "July", "August",
    "September", "October", "November", "December"
]

df["MONTH_NAME"] = pd.Categorical(
    df["MONTH_NAME"],
    categories=month_order,
    ordered=True
)

df.head()

,DATE,HOLIDAY,WEEKDAY,WEATHERSIT,TEMP,ATEMP,HUMIDITY,WINDSPEED,CASUAL,REGISTERED,COUNT,MONTH,MONTH_NAME,YEAR,BADWEATHER
0,2011-01-01,NO,NO,2,11.0,11.0,81.0,17.0,331,654,985,1,January,2011,NO
1,2011-01-02,NO,NO,2,9.0,6.5,71.5,17.0,131,670,801,1,January,2011,NO
2,2011-01-03,NO,YES,1,1.0,4.0,44.0,18.0,120,1229,1349,1,January,2011,NO
3,2011-01-04,NO,YES,1,2.0,2.5,64.0,9.0,108,1454,1562,1,January,2011,NO
4,2011-01-05,NO,YES,1,2.5,1.0,42.5,13.0,82,1518,1600,1,January,2011,NO


In [63]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print(f"Date range: {df['DATE'].min().date()} to {df['DATE'].max().date()}")
print(f"Missing values: {df.isna().sum().sum()}")

df.info()

Rows: 731
Columns: 15
Date range: 2011-01-01 to 2012-12-31
Missing values: 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 731 entries, 0 to 730
Data columns (total 15 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   DATE        731 non-null    datetime64[ns]
 1   HOLIDAY     731 non-null    object        
 2   WEEKDAY     731 non-null    object        
 3   WEATHERSIT  731 non-null    int64         
 4   TEMP        731 non-null    float64       
 5   ATEMP       731 non-null    float64       
 6   HUMIDITY    731 non-null    float64       
 7   WINDSPEED   731 non-null    float64       
 8   CASUAL      731 non-null    int64         
 9   REGISTERED  731 non-null    int64         
 10  COUNT       731 non-null    int64         
 11  MONTH       731 non-null    int32         
 12  MONTH_NAME  731 non-null    category      
 13  YEAR        731 non-null    int32         
 14  BADWEATHER  731 non-null    object        
dt

# Exploratory Data Analysis

This section explores ridership patterns, seasonality, weather effects, and rider behavior before developing predictive models.

Key questions include:

- How does demand vary throughout the year?
- How do registered and casual riders differ?
- What effect does weather have on demand?
- Which variables appear most strongly related to ridership?

In [64]:
df.describe().round(2)

,DATE,WEATHERSIT,TEMP,ATEMP,HUMIDITY,WINDSPEED,CASUAL,REGISTERED,COUNT,MONTH,YEAR
count,731,731.00,731.00,731.00,731.00,731.00,731.00,731.00,731.00,731.00,731.0
mean,2012-01-01 00:00:00,1.40,15.87,16.00,63.17,12.82,848.18,3656.17,4504.35,6.52,2011.5
min,2011-01-01 00:00:00,1.00,1.00,1.00,17.00,0.00,2.00,20.00,22.00,1.00,2011.0
25%,2011-07-02 12:00:00,1.00,8.00,6.60,51.00,9.00,315.50,2497.00,3152.00,4.00,2011.0
50%,2012-01-01 00:00:00,1.00,16.00,16.00,62.00,12.00,713.00,3662.00,4548.00,7.00,2012.0
75%,2012-07-01 12:00:00,2.00,23.15,23.95,74.00,16.00,1096.00,4776.50,5956.00,10.00,2012.0
max,2012-12-31 00:00:00,3.00,34.00,41.00,100.00,40.16,3410.00,6946.00,8714.00,12.00,2012.0
std,NaN,0.54,8.83,9.67,15.47,5.54,686.62,1560.26,1937.21,3.45,0.5


In [65]:
print(f"Observations: {len(df):,}")
print(f"Study period: {df.DATE.min().date()} to {df.DATE.max().date()}")

print("\nAverage Daily Rentals")
print(f"Total: {df.COUNT.mean():,.0f}")
print(f"Registered: {df.REGISTERED.mean():,.0f}")
print(f"Casual: {df.CASUAL.mean():,.0f}")

Observations: 731
Study period: 2011-01-01 to 2012-12-31

Average Daily Rentals
Total: 4,504
Registered: 3,656
Casual: 848


In [66]:
monthly = (
    df.groupby("MONTH_NAME")["COUNT"]
      .mean()
      .reset_index()
)

fig = px.bar(
    monthly,
    x="MONTH_NAME",
    y="COUNT",
    title="Average Daily Bike Rentals by Month",
    labels={
        "MONTH_NAME":"Month",
        "COUNT":"Average Daily Rentals"
    }
)

fig.update_layout(xaxis_tickangle=-45)

fig.show()

/tmp/ipykernel_58/4025421221.py:2: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [67]:
monthly_users = (
    df.groupby("MONTH_NAME")[["REGISTERED","CASUAL"]]
      .mean()
      .reset_index()
)

fig = px.bar(
    monthly_users,
    x="MONTH_NAME",
    y=["REGISTERED","CASUAL"],
    barmode="group",
    title="Average Daily Rentals by Rider Type"
)

fig.update_layout(xaxis_tickangle=-45)

fig.show()

/tmp/ipykernel_58/2308933869.py:2: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [68]:
fig = px.line(
    df,
    x="DATE",
    y="COUNT",
    title="Daily Bike Rentals Over Time"
)

fig.show()

In [69]:
df["Rolling30"] = (
    df["COUNT"]
      .rolling(30)
      .mean()
)

fig = px.line(
    df,
    x="DATE",
    y=["COUNT","Rolling30"],
    title="Daily Ridership with 30-Day Rolling Average"
)

fig.show()

In [70]:
fig = px.box(
    df,
    x="BADWEATHER",
    y="COUNT",
    color="BADWEATHER",
    title="Impact of Weather on Daily Rentals"
)

fig.show()

In [71]:
fig = px.scatter(
    df,
    x="ATEMP",
    y="COUNT",
    color="BADWEATHER",
    trendline="ols",
    title="Feels-Like Temperature (C) vs Daily Rentals"
)

fig.show()

In [72]:
import plotly.figure_factory as ff

corr = (
    df[
        [
            "COUNT",
            "REGISTERED",
            "CASUAL",
            "TEMP",
            "ATEMP",
            "HUMIDITY",
            "WINDSPEED"
        ]
    ]
    .corr()
    .round(2)
)

fig = ff.create_annotated_heatmap(
    z=corr.values,
    x=list(corr.columns),
    y=list(corr.index),
    annotation_text=corr.values,
    colorscale="Viridis"
)

fig.update_layout(title="Correlation Matrix")

fig.show()

## Exploratory Analysis Summary

Key findings include:

- Bike demand demonstrates strong seasonal variation, peaking during warmer months.
- Registered riders consistently account for the majority of rentals throughout the year.
- Daily rentals increase with perceived temperature until extremely hot conditions.
- Bad weather is associated with substantially lower ridership.
- Temperature and "feels like" temperature are highly correlated, suggesting potential multicollinearity that should be considered during model development.

# Feature Engineering & Target Leakage

Before developing predictive models, additional features are created to improve analysis and model performance.

During exploratory modeling, it is also important to identify situations where information from the target variable unintentionally appears among the predictors. This issue, known as **target leakage**, can produce unrealistically accurate models that would fail in real-world forecasting.

The following section demonstrates this phenomenon before constructing a valid predictive model.

In [73]:
model_df = df.copy()

model_df.head()

,DATE,HOLIDAY,WEEKDAY,WEATHERSIT,TEMP,ATEMP,HUMIDITY,WINDSPEED,CASUAL,REGISTERED,COUNT,MONTH,MONTH_NAME,YEAR,BADWEATHER,Rolling30
0,2011-01-01,NO,NO,2,11.0,11.0,81.0,17.0,331,654,985,1,January,2011,NO,NaN
1,2011-01-02,NO,NO,2,9.0,6.5,71.5,17.0,131,670,801,1,January,2011,NO,NaN
2,2011-01-03,NO,YES,1,1.0,4.0,44.0,18.0,120,1229,1349,1,January,2011,NO,NaN
3,2011-01-04,NO,YES,1,2.0,2.5,64.0,9.0,108,1454,1562,1,January,2011,NO,NaN
4,2011-01-05,NO,YES,1,2.5,1.0,42.5,13.0,82,1518,1600,1,January,2011,NO,NaN


In [74]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

X = model_df[["REGISTERED", "CASUAL"]]
y = model_df["COUNT"]

model = LinearRegression()
model.fit(X, y)

pred = model.predict(X)

print(f"R²: {r2_score(y, pred):.4f}")

R²: 1.0000


Why Did the Model Perform Perfectly?

The model achieved an R² value of approximately 1.00 because it was provided with the exact components used to calculate the target variable.

Daily ridership (COUNT) is defined as:

COUNT = REGISTERED + CASUAL

Since the predictors already contain the complete answer, the model is not learning relationships between weather, seasonality, or rider behavior. Instead, it is simply reconstructing the target variable.

This is known as target leakage and represents a common modeling mistake that produces misleadingly high performance.

In a real forecasting scenario, the number of registered and casual riders would not be known before predicting daily demand. Therefore, these variables must be excluded from all legitimate predictive models.

In [75]:
predictors = [
    "TEMP",
    "ATEMP",
    "HUMIDITY",
    "WINDSPEED",
    "HOLIDAY",
    "WEEKDAY",
    "WEATHERSIT",
    "MONTH",
    "YEAR"
]

model_data = model_df[predictors + ["COUNT"]]

model_data.head()

,TEMP,ATEMP,HUMIDITY,WINDSPEED,HOLIDAY,WEEKDAY,WEATHERSIT,MONTH,YEAR,COUNT
0,11.0,11.0,81.0,17.0,NO,NO,2,1,2011,985
1,9.0,6.5,71.5,17.0,NO,NO,2,1,2011,801
2,1.0,4.0,44.0,18.0,NO,YES,1,1,2011,1349
3,2.0,2.5,64.0,9.0,NO,YES,1,1,2011,1562
4,2.5,1.0,42.5,13.0,NO,YES,1,1,2011,1600


# Predictive Modeling

Following exploratory analysis and the removal of target leakage, a linear regression model is developed to predict daily bike rentals using weather, calendar, and seasonal variables.

The objective is not only to produce accurate predictions, but also to understand which factors have the greatest influence on daily demand. The model will be evaluated using a train/test split and several standard regression performance metrics.

In [76]:
from sklearn.model_selection import train_test_split

# Define predictors and target
X = model_data.drop(columns="COUNT")
y = model_data["COUNT"]

# One-hot encode categorical variables
X = pd.get_dummies(
    X,
    columns=["HOLIDAY", "WEEKDAY", "WEATHERSIT"],
    drop_first=True
)

X.head()

,TEMP,ATEMP,HUMIDITY,WINDSPEED,MONTH,YEAR,HOLIDAY_YES,WEEKDAY_YES,WEATHERSIT_2,WEATHERSIT_3
0,11.0,11.0,81.0,17.0,1,2011,False,False,True,False
1,9.0,6.5,71.5,17.0,1,2011,False,False,True,False
2,1.0,4.0,44.0,18.0,1,2011,False,True,False,False
3,2.0,2.5,64.0,9.0,1,2011,False,True,False,False
4,2.5,1.0,42.5,13.0,1,2011,False,True,False,False


## Preparing the Modeling Dataset

Several variables in the dataset represent categorical information (such as weekday and weather category). These variables are converted into dummy variables so they can be used within a linear regression model.

The target variable is total daily bike rentals (`COUNT`).

In [77]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training observations:", len(X_train))
print("Testing observations:", len(X_test))

Training observations: 584
Testing observations: 147


## Training and Testing Data

The dataset is divided into training and testing subsets using an 80/20 split.

The model is trained using the training data and evaluated on observations it has never seen before. This provides a more realistic estimate of predictive performance than evaluating the model on the same data used for training.

In [78]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()

model.fit(X_train, y_train)

predictions = model.predict(X_test)

In [79]:
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

r2 = r2_score(y_test, predictions)
mae = mean_absolute_error(y_test, predictions)
rmse = mean_squared_error(y_test, predictions) ** 0.5

print(f"R²:   {r2:.3f}")
print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")

R²:   0.814
MAE:  667.95
RMSE: 863.22


## Full Model Performance

The full linear regression model explains approximately **81.4% of the variation** in daily bicycle rentals within the test data.

The model's mean absolute error is approximately **668 rentals per day**, while its RMSE is approximately **863 rentals per day**. The larger RMSE indicates that some individual days produce substantially greater errors than the average prediction.

These results demonstrate that weather, calendar, and seasonal variables provide strong predictive information. However, model diagnostics reveal that actual and perceived temperature contain highly overlapping information, which reduces the stability and interpretability of their individual coefficients.

In [80]:
import plotly.express as px

results = pd.DataFrame({
    "Actual": y_test,
    "Predicted": predictions
})

fig = px.scatter(
    results,
    x="Actual",
    y="Predicted",
    trendline="ols",
    title="Actual vs Predicted Daily Rentals"
)

fig.show()

## Actual vs. Predicted Values

This visualization compares the model's predictions with the observed number of daily rentals.

Points located near the diagonal trend indicate accurate predictions, while greater dispersion suggests areas where prediction accuracy could be improved.

In [81]:
results["Residual"] = (
    results["Actual"] -
    results["Predicted"]
)

fig = px.scatter(
    results,
    x="Predicted",
    y="Residual",
    title="Residual Plot"
)

fig.add_hline(y=0)

fig.show()

## Residual Diagnostics

Residuals represent the difference between actual and predicted values.

Ideally, residuals should appear randomly scattered around zero. Visible patterns or systematic trends may indicate that important variables or nonlinear relationships are not being captured by the model.

In [82]:
coefficients = (
    pd.DataFrame({
        "Feature": X.columns,
        "Coefficient": model.coef_
    })
    .sort_values(
        "Coefficient",
        key=abs,
        ascending=False
    )
)

coefficients.head(15)

,Feature,Coefficient
5,YEAR,1985.580606
9,WEATHERSIT_3,-1770.134088
6,HOLIDAY_YES,-827.535189
8,WEATHERSIT_2,-499.426666
0,TEMP,240.035164
7,WEEKDAY_YES,158.930399
1,ATEMP,-115.445295
4,MONTH,101.819116
3,WINDSPEED,-35.456447
2,HUMIDITY,-7.902231


In [83]:
fig = px.bar(
    coefficients.head(15),
    x="Coefficient",
    y="Feature",
    orientation="h",
    title="Most Influential Model Coefficients"
)

fig.show()

## Interpreting Model Coefficients

Model coefficients estimate the direction and relative magnitude of each predictor's relationship with daily bike rentals while holding all other variables constant.

Positive coefficients indicate variables associated with increased demand, whereas negative coefficients indicate variables associated with reduced demand.

In [84]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import numpy as np
import pandas as pd

# Convert every predictor to a numeric float type
X_vif = X.astype(float).copy()

# Verify there are no infinite or missing values
X_vif = X_vif.replace([np.inf, -np.inf], np.nan)

print("Missing values:", X_vif.isna().sum().sum())
print("Data types:")
print(X_vif.dtypes.value_counts())

# Calculate VIF
vif = pd.DataFrame({
    "Variable": X_vif.columns,
    "VIF": [
        variance_inflation_factor(X_vif.values, i)
        for i in range(X_vif.shape[1])
    ]
})

vif.sort_values("VIF", ascending=False).head(15)

Missing values: 0
Data types:
float64    10
Name: count, dtype: int64


,Variable,VIF
0,TEMP,183.120485
1,ATEMP,161.171563
5,YEAR,47.687943
2,HUMIDITY,34.082636
3,WINDSPEED,7.737369
4,MONTH,5.224745
7,WEEKDAY_YES,3.551418
8,WEATHERSIT_2,2.354989
9,WEATHERSIT_3,1.414558
6,HOLIDAY_YES,1.046717


## Multicollinearity Assessment

Variance Inflation Factor (VIF) analysis was performed to evaluate correlation among predictor variables.

The results indicate substantial multicollinearity between **temperature (TEMP)** and **feels-like temperature (ATEMP)**. This outcome is expected because both variables measure closely related weather conditions.

Similarly, elevated VIF values for **YEAR** and **HUMIDITY** suggest these variables share information with other seasonal and weather-related predictors.

While multicollinearity does not necessarily reduce predictive performance, it can make individual regression coefficients unstable and more difficult to interpret. For this reason, a reduced model is developed and compared against the full model to determine whether similar predictive performance can be achieved with fewer correlated predictors.

In [85]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Convert all predictors to numeric floats
X_vif = X.astype(float).copy()

# Add an intercept for a standard VIF calculation
X_vif_constant = sm.add_constant(X_vif)

vif_corrected = pd.DataFrame({
    "Variable": X_vif_constant.columns,
    "VIF": [
        variance_inflation_factor(X_vif_constant.values, i)
        for i in range(X_vif_constant.shape[1])
    ]
})

vif_corrected = (
    vif_corrected[
        vif_corrected["Variable"] != "const"
    ]
    .sort_values("VIF", ascending=False)
    .reset_index(drop=True)
)

vif_corrected

,Variable,VIF
0,TEMP,43.223295
1,ATEMP,43.093813
2,HUMIDITY,1.958426
3,WEATHERSIT_2,1.566417
4,WEATHERSIT_3,1.373924
5,WINDSPEED,1.218485
6,MONTH,1.142984
7,YEAR,1.024203
8,WEEKDAY_YES,1.020276
9,HOLIDAY_YES,1.016702


## Corrected Multicollinearity Assessment

The VIF calculation was repeated with an intercept included in the regression design matrix. Including an intercept produces the standard centered VIF calculation and avoids artificially inflating values for variables whose means are far from zero.

The revised results are used to identify predictors that contain substantially overlapping information. Particular attention is given to `TEMP` and `ATEMP`, which represent actual and perceived temperature and are expected to be strongly correlated.

In [86]:
temp_correlation = df[["TEMP", "ATEMP"]].corr()

temp_correlation

,TEMP,ATEMP
TEMP,1.000000,0.987967
ATEMP,0.987967,1.000000


In [87]:
fig = px.scatter(
    df,
    x="TEMP",
    y="ATEMP",
    trendline="ols",
    title="Relationship Between Actual and Feels-Like Temperature",
    labels={
        "TEMP": "Actual Temperature",
        "ATEMP": "Feels-Like Temperature"
    }
)

fig.show()

## Actual Temperature vs. Feels-Like Temperature

This visualization examines the relationship between actual temperature (`TEMP`) and perceived or feels-like temperature (`ATEMP`).

The points follow a nearly linear pattern, indicating that the two variables contain very similar information. Including both variables in the same regression model makes it difficult to isolate their individual effects and can lead to unstable coefficient estimates.

Because perceived temperature may more closely reflect how weather influences a person's decision to ride a bicycle, `ATEMP` is retained while `TEMP` is removed from the reduced model.

In [88]:
# Remove actual temperature while retaining feels-like temperature
X_reduced = X.drop(columns=["TEMP"])

print("Full model predictors:", X.shape[1])
print("Reduced model predictors:", X_reduced.shape[1])

X_reduced.head()

Full model predictors: 10
Reduced model predictors: 9


,ATEMP,HUMIDITY,WINDSPEED,MONTH,YEAR,HOLIDAY_YES,WEEKDAY_YES,WEATHERSIT_2,WEATHERSIT_3
0,11.0,81.0,17.0,1,2011,False,False,True,False
1,6.5,71.5,17.0,1,2011,False,False,True,False
2,4.0,44.0,18.0,1,2011,False,True,False,False
3,2.5,64.0,9.0,1,2011,False,True,False,False
4,1.0,42.5,13.0,1,2011,False,True,False,False


## Reduced Predictor Set

The reduced model excludes `TEMP` while retaining `ATEMP`.

These variables measure closely related temperature conditions, and their simultaneous inclusion creates substantial multicollinearity. Retaining only perceived temperature reduces redundancy while preserving the weather information that is likely most relevant to rider behavior.

In [89]:
# Use the existing train/test indices to ensure a fair comparison
X_reduced_train = X_reduced.loc[X_train.index]
X_reduced_test = X_reduced.loc[X_test.index]

print("Reduced training observations:", len(X_reduced_train))
print("Reduced testing observations:", len(X_reduced_test))

Reduced training observations: 584
Reduced testing observations: 147


## Consistent Model Evaluation

The reduced model uses the same training and testing observations as the full model.

Holding the data split constant ensures that any differences in performance are caused by the predictor set rather than by evaluating the models on different samples.

In [90]:
reduced_model = LinearRegression()

reduced_model.fit(
    X_reduced_train,
    y_train
)

reduced_predictions = reduced_model.predict(
    X_reduced_test
)

## Reduced Linear Regression Model

A second linear regression model is trained after removing the redundant actual-temperature variable.

The objective is to determine whether a simpler model can preserve predictive performance while improving coefficient stability and interpretability.

In [91]:
reduced_r2 = r2_score(
    y_test,
    reduced_predictions
)

reduced_mae = mean_absolute_error(
    y_test,
    reduced_predictions
)

reduced_rmse = mean_squared_error(
    y_test,
    reduced_predictions
) ** 0.5

print(f"Reduced Model R²:   {reduced_r2:.3f}")
print(f"Reduced Model MAE:  {reduced_mae:.2f}")
print(f"Reduced Model RMSE: {reduced_rmse:.2f}")

Reduced Model R²:   0.801
Reduced Model MAE:  690.49
Reduced Model RMSE: 892.61


## Reduced Model Performance

The reduced linear regression model explains approximately **80.1% of the variation** in unseen daily ridership observations.

Its mean absolute error is approximately **690 rentals per day**, meaning that predictions differ from observed demand by about 690 rentals on an average test-set day. Its RMSE of approximately **893 rentals** indicates that several days contain larger prediction errors.

Although the reduced model performs slightly worse than the full model, it retains substantial predictive power while eliminating the severe multicollinearity caused by including both actual and perceived temperature..

In [92]:
model_comparison = pd.DataFrame({
    "Model": [
        "Full Linear Regression",
        "Reduced Linear Regression"
    ],
    "Predictors": [
        X.shape[1],
        X_reduced.shape[1]
    ],
    "R²": [
        r2,
        reduced_r2
    ],
    "MAE": [
        mae,
        reduced_mae
    ],
    "RMSE": [
        rmse,
        reduced_rmse
    ]
})

model_comparison.round({
    "R²": 3,
    "MAE": 2,
    "RMSE": 2
})

,Model,Predictors,R²,MAE,RMSE
0,Full Linear Regression,10,0.814,667.95,863.22
1,Reduced Linear Regression,9,0.801,690.49,892.61


# Final Model Assessment

The full linear regression model produced the strongest predictive results:

- **R²:** 0.814
- **MAE:** 667.95 daily rentals
- **RMSE:** 863.22 daily rentals

The reduced model, which removed actual temperature (`TEMP`), produced slightly weaker results:

- **R²:** 0.801
- **MAE:** 690.49 daily rentals
- **RMSE:** 892.61 daily rentals

Removing `TEMP` reduced R² by approximately 0.013 and increased the typical prediction error by about 23 rentals per day. The increase in RMSE was approximately 29 rentals.

The full model therefore provides marginally better predictive accuracy. However, its VIF results show substantial multicollinearity between actual temperature and perceived temperature, with both variables producing VIF values above 43. In the reduced model, every VIF value falls below 2, indicating that the multicollinearity issue has been effectively resolved.

Because the purpose of this analysis includes understanding the drivers of demand—not only maximizing predictive accuracy—the **reduced model is selected as the preferred explanatory model**. It sacrifices relatively little predictive performance while offering substantially clearer and more stable coefficient interpretation.

For a production forecasting system focused strictly on predictive accuracy, the full model could still be considered, provided that coefficient interpretation is not the primary objective.

In [93]:
comparison_long = model_comparison.melt(
    id_vars=["Model", "Predictors"],
    value_vars=["MAE", "RMSE"],
    var_name="Metric",
    value_name="Error"
)

fig = px.bar(
    comparison_long,
    x="Metric",
    y="Error",
    color="Model",
    barmode="group",
    title="Prediction Error by Model",
    labels={
        "Error": "Daily Rental Error",
        "Metric": "Performance Metric"
    }
)

fig.show()

## Prediction Error Comparison

This visualization compares the MAE and RMSE values of the full and reduced linear regression models.

Lower values indicate better predictive accuracy. MAE represents the typical prediction error, while RMSE is more sensitive to days with especially large errors.

Similar error levels would indicate that removing the redundant temperature variable does not materially weaken the model.

In [94]:
reduced_results = pd.DataFrame({
    "Actual": y_test,
    "Predicted": reduced_predictions
})

fig = px.scatter(
    reduced_results,
    x="Actual",
    y="Predicted",
    trendline="ols",
    title="Reduced Model: Actual vs. Predicted Daily Rentals",
    labels={
        "Actual": "Actual Daily Rentals",
        "Predicted": "Predicted Daily Rentals"
    }
)

# Add a perfect-prediction reference line
minimum_value = min(
    reduced_results["Actual"].min(),
    reduced_results["Predicted"].min()
)

maximum_value = max(
    reduced_results["Actual"].max(),
    reduced_results["Predicted"].max()
)

fig.add_shape(
    type="line",
    x0=minimum_value,
    y0=minimum_value,
    x1=maximum_value,
    y1=maximum_value,
    line=dict(dash="dash")
)

fig.show()

## Reduced Model Predictions

This visualization compares predicted daily rentals with the actual observed values for the reduced model.

The dashed diagonal line represents perfect prediction. Points close to this line indicate accurate estimates, while points farther away represent larger prediction errors.

The overall pattern shows whether the model performs consistently across low-, moderate-, and high-demand days.

In [95]:
reduced_results["Residual"] = (
    reduced_results["Actual"]
    - reduced_results["Predicted"]
)

fig = px.scatter(
    reduced_results,
    x="Predicted",
    y="Residual",
    title="Reduced Model Residuals",
    labels={
        "Predicted": "Predicted Daily Rentals",
        "Residual": "Residual"
    }
)

fig.add_hline(
    y=0,
    line_dash="dash"
)

fig.show()

## Reduced Model Residuals

Residuals measure the difference between actual and predicted daily rentals.

A well-specified linear regression model should produce residuals that are randomly distributed around zero. Patterns, curves, or increasing dispersion may indicate that the relationship is nonlinear, that error variance changes with demand, or that important predictors are missing.

Large positive residuals represent days when demand was substantially higher than predicted, while large negative residuals represent days when the model overestimated demand.

In [96]:
X_reduced_vif = X_reduced.astype(float).copy()
X_reduced_vif_constant = sm.add_constant(X_reduced_vif)

reduced_vif = pd.DataFrame({
    "Variable": X_reduced_vif_constant.columns,
    "VIF": [
        variance_inflation_factor(
            X_reduced_vif_constant.values,
            i
        )
        for i in range(X_reduced_vif_constant.shape[1])
    ]
})

reduced_vif = (
    reduced_vif[
        reduced_vif["Variable"] != "const"
    ]
    .sort_values("VIF", ascending=False)
    .reset_index(drop=True)
)

reduced_vif

,Variable,VIF
0,HUMIDITY,1.957589
1,WEATHERSIT_2,1.563845
2,WEATHERSIT_3,1.373875
3,WINDSPEED,1.215267
4,MONTH,1.109564
5,ATEMP,1.106582
6,YEAR,1.023878
7,WEEKDAY_YES,1.019691
8,HOLIDAY_YES,1.016631


In [97]:
# Create a coefficient table for the reduced model
reduced_coefficients = pd.DataFrame({
    "Feature": X_reduced.columns,
    "Coefficient": reduced_model.coef_
})

# Sort by absolute coefficient magnitude
reduced_coefficients["Absolute_Coefficient"] = (
    reduced_coefficients["Coefficient"].abs()
)

reduced_coefficients = (
    reduced_coefficients
    .sort_values("Absolute_Coefficient", ascending=True)
)

fig = px.bar(
    reduced_coefficients,
    x="Coefficient",
    y="Feature",
    orientation="h",
    title="Reduced Model Coefficients",
    labels={
        "Coefficient": "Estimated Change in Daily Rentals",
        "Feature": "Predictor"
    }
)

fig.add_vline(
    x=0,
    line_dash="dash"
)

fig.show()

## Reduced Model Coefficients

This visualization displays the estimated relationship between each predictor and daily bicycle demand while holding the remaining variables constant.

Positive coefficients are associated with increased predicted ridership, while negative coefficients are associated with reduced predicted ridership. The largest effects are associated with the study year and adverse weather categories, demonstrating that both overall system growth and daily weather conditions are important drivers of demand.

Coefficient magnitude should be interpreted cautiously because predictors use different measurement scales. The chart is therefore most useful for understanding the direction of each relationship and identifying variables with especially large estimated effects.

# Business Recommendations

## 1. Use weather forecasts to support daily fleet planning

Weather conditions are strongly related to ridership, and adverse weather categories have large negative model coefficients. Capital Bikeshare can incorporate short-term weather forecasts into bicycle allocation and staffing decisions.

On favorable-weather days, additional bicycles should be positioned at historically high-demand stations. During poor weather, operations teams can reduce unnecessary redistribution activity and redirect resources toward maintenance.

## 2. Increase operational capacity during peak riding months

Exploratory analysis shows a pronounced seasonal pattern, with demand increasing during warmer months and declining during winter.

Capital Bikeshare should schedule preventive maintenance and larger equipment projects during lower-demand periods so that more bicycles are available during the spring, summer, and early fall.

## 3. Develop separate strategies for registered and casual riders

Registered riders account for most daily rentals, while casual ridership is more sensitive to seasonal and weather conditions.

Retention and service-reliability efforts should focus on registered riders, who provide a comparatively stable demand base. Seasonal promotions and membership-conversion campaigns can target casual riders during warmer months.

## 4. Plan for continued system growth

The strong positive year coefficient indicates that ridership was substantially higher in 2012 than in 2011 after controlling for the other model variables.

Capacity planning should account for growth in the overall customer base rather than relying only on historical seasonal averages. Fleet size, docking capacity, maintenance resources, and redistribution staffing may all need to increase as adoption grows.

## 5. Investigate unusually high-error days

The residual plots show that the model does not predict every day equally well. Days with unusually large residuals may reflect special events, precipitation intensity, station outages, transit disruptions, or other factors not included in the dataset.

Reviewing these high-error days could identify additional predictors that improve future demand forecasts.

# Limitations

This analysis has several limitations that should be considered when interpreting the findings.

- The dataset contains only two years of daily observations, which limits the ability to evaluate longer-term demand patterns.
- The model uses system-wide daily totals and cannot identify demand differences among individual stations or neighborhoods.
- Weather is represented through broad categories and several continuous measures, but the dataset does not include detailed precipitation timing or intensity.
- Special events, tourism activity, transit disruptions, bicycle availability, station capacity, and pricing changes are not included.
- The random train/test split evaluates predictions on unseen observations but does not fully reproduce a future forecasting environment. A chronological split would provide a more rigorous test of forecasting performance.
- Linear regression assumes additive and approximately linear relationships. Some demand patterns may be nonlinear or involve interactions among weather, season, and calendar variables.
- The model identifies statistical relationships but does not establish that the predictors directly cause changes in ridership.

# Future Improvements

Future versions of this analysis could improve predictive performance and operational relevance by:

- Evaluating the model with a chronological train/test split.
- Incorporating station-level rental and bicycle-availability data.
- Adding detailed precipitation, snowfall, visibility, and extreme-weather measures.
- Including local events, holidays, tourism activity, and transit disruptions.
- Testing nonlinear models such as Random Forest, Gradient Boosting, or XGBoost.
- Using time-series cross-validation to evaluate performance across multiple forecasting periods.
- Investigating interaction effects between season, weekday status, and weather conditions.
- Developing separate models for registered and casual riders.

# Conclusion

This analysis examined the weather, calendar, and seasonal factors associated with daily Capital Bikeshare demand and developed linear regression models to predict total ridership.

The full model explained approximately **81.4%** of variation in unseen test observations, with an average absolute prediction error of approximately **668 rentals per day**. Model diagnostics identified severe multicollinearity between actual temperature and perceived temperature.

A reduced model excluding actual temperature explained approximately **80.1%** of test-set variation, with an average absolute error of approximately **690 rentals per day**. Although its predictive accuracy was slightly lower, the reduced model eliminated the multicollinearity issue and provided a more interpretable representation of demand drivers.

The results show that ridership is shaped by seasonality, weather conditions, perceived temperature, and broader growth in system use. These findings can support fleet allocation, maintenance scheduling, customer segmentation, and operational planning.

Most importantly, the analysis demonstrates that model quality should not be evaluated through predictive metrics alone. Data leakage, multicollinearity, interpretability, and the practical availability of predictor variables must also be considered when selecting a model for business use.